# Molecular Dynamics of Lennard-Jones charged particles — Verlet integrator

*Utrecht University Molecular Modelling courses from the [Bonvin lab](https://bonvinlab.org).*

This notebook runs a simple **molecular dynamics (MD)** simulation of a 2D system of
Lennard-Jones particles that may also carry a charge (Coulomb interaction). Where the
companion *energy-minimization* notebooks only walk the system **downhill** to a nearby
minimum, MD instead **integrates Newton's equations of motion** in time: the particles are
given initial velocities drawn from a Maxwell-Boltzmann distribution at a chosen temperature,
and then move under the forces they exert on one another. The system therefore *explores* the
energy surface — climbing barriers and sampling many configurations — rather than freezing
into the first minimum it finds.

## Theory in brief

### Lennard-Jones

$$E_{LJ}(r) \;=\; 4\varepsilon\left[\left(\frac{\sigma}{r}\right)^{12}-\left(\frac{\sigma}{r}\right)^{6}\right]$$

The $(\sigma/r)^{12}$ term is the steep short-range **repulsion** (overlapping atoms), the
$-(\sigma/r)^{6}$ term the weaker long-range **attraction**. The two cancel exactly at
$r=\sigma$ (`Sigma`), and the energy reaches its minimum $-\varepsilon$ (`Epsilon`) a little
further out, at $R_{min}=2^{1/6}\sigma\approx1.12\,\sigma$. So `Sigma` sets the particle
*size* and `Epsilon` the well *depth*. The code evaluates this same expression from the
**squared** distance, so the inner loop needs no square root.

*(For the potential on its own — the two components plotted separately, in real force-field
units — see [`LJ-ELEC_Potentials`](LJ-ELEC_Potentials.ipynb).)*

### Coulomb

$$E_{Coul} = \frac{q_a q_b}{\epsilon_r\, r}$$

Like charges repel ($E>0$), unlike charges attract ($E<0$); the dielectric constant
`Dielec` ($\epsilon_r$) screens (weakens) the interaction.

### Verlet integration
The **force** on each atom is minus the gradient of the total potential energy. Given the
force, Newton's law $\mathbf{F} = m\,\mathbf{a}$ is integrated with the **Verlet** algorithm,
which is remarkable for needing no velocities at all: it propagates positions using the
current and *previous* positions together with the current force. **Deriving that update rule
is your first exercise** -- see section 5, where you will also write the code that implements it.

The very first step has no previous position to work from, so it has to be bootstrapped from
the initial velocities with a single Taylor step -- yours to derive and implement as well.
Velocities are then recovered from successive positions by a central difference, and give the
**kinetic energy**
$E_{kin} = \tfrac12 m \sum v_i^2$ and hence the instantaneous **temperature**
($E_{kin} = N k_B T$ in 2D). In a good simulation the **total** energy $E_{tot}=E_{pot}+E_{kin}$
is conserved while potential and kinetic energy flow back and forth; the time step `timestep`
must be small enough for this to hold.

## 1. Imports

The numerical core uses only the Python **standard library** (`math`, `random`), so it runs
on a bare Python install. **matplotlib** is the one third-party dependency — it draws the
static figures and the trajectory animation (embedded inline as interactive HTML via
`jshtml`). The cell below first **installs matplotlib if it is missing** (handy on Google
Colab), then imports everything; `%matplotlib inline` renders figures inside the notebook.

In [ ]:
# --- Install required packages (works locally, on Colab, and on JupyterLite) ---
%pip install -q matplotlib

from math import sqrt, log, sin, cos

import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from matplotlib.animation import FuncAnimation
from matplotlib import rc

from random import random, randint, seed

# Show animations inline
rc('animation', html='jshtml')
%matplotlib inline

## 2. Helper functions

Small utilities used throughout:

* `dist` / `dist2` — Euclidean distance and its square (the squared form avoids a needless
  `sqrt` when we only need to compare distances).
* `SignR(a, b)` — returns `a` with the sign of `b`. It implements the **minimum-image
  convention**: the combination `tmp - SignR(halfbox, tmp-halfbox) - SignR(halfbox, tmp+halfbox)`
  wraps a coordinate difference into the range $[-\tfrac{box}{2}, +\tfrac{box}{2}]$, so each
  atom interacts with the *nearest periodic copy* of its neighbours (periodic boundary
  conditions).
* `charge_color` — purely cosmetic: white for positive charges, dark for negative, used when
  drawing the particles.

In [ ]:
### distance ###
def dist(A, B):
    return sqrt((A[0]-B[0])**2 + (A[1]-B[1])**2)

### squared distance ###
def dist2(A, B):
    return (A[0]-B[0])**2 + (A[1]-B[1])**2

### change sign ###
def SignR(a, b):
    if b > 0:
        return a
    else:
        return -a

### colour particles based on charge ###
def charge_color(charge, qat):
    if charge == qat:
        return "#FFFFFF"   # positive
    else:
        return "#333333"   # negative

## 3. Energy and temperature functions

The **potential** energy is the sum over all particle pairs of the Lennard-Jones and Coulomb
contributions, using the **nearest image** convention (periodic boundary conditions).
Distances are handled as **squared** distances (`distsquare`) throughout the inner loops:
this avoids a `sqrt` for every pair and lets the cutoff test (`distsquare < cutoffsquare`)
skip distant pairs cheaply. A `sqrt` is taken only where a term actually needs $r$ (Coulomb).

`calc_temp` turns the velocities into the **kinetic** energy $E_{kin}=\tfrac12 m\sum v_i^2$
and the instantaneous temperature via $E_{kin}=N k_B T$.

In [ ]:
# LJ energy from the squared distance
def LJ2(distsquare, epsilon, sigma_exp6):
    # E_LJ = 4 eps [ (sigma/r)^12 - (sigma/r)^6 ], from the squared distance
    u = (1/distsquare)**3 * sigma_exp6        # u = (sigma/r)^6,  u*u = (sigma/r)^12
    return 4*epsilon * u * (u - 1)            # = 4 eps [ (sigma/r)^12 - (sigma/r)^6 ]

# classical Coulomb from the squared distance
def Coulomb2(r, dielec, qa, qb):
    return qa*qb / (dielec*sqrt(r))

# Total potential energy Evdw + Ecoulomb (uses squared distance), with periodic boundary conditions
def Calc_Ene2(coord, epsilon, sigma, dielec, cutoffsquare, boxdim, elec=1):
    Ene = 0.0
    ELJ = 0.0
    ECoul = 0.0
    sigma_exp6 = sigma**6
    # doubly nested loop over all particle pairs
    for i in range(len(coord)-1):
        for j in range(i+1, len(coord)):
            # squared atomic distance (nearest image)
            distsquare = 0
            for k in range(2):
                tmp = coord[j][k] - coord[i][k]
                halfbox = boxdim[k]/2
                tmp = tmp - SignR(halfbox, tmp-halfbox) - SignR(halfbox, tmp+halfbox)
                distsquare += tmp**2
            if distsquare < cutoffsquare:
                qa = coord[i][2]
                qb = coord[j][2]
                vdw = LJ2(distsquare, epsilon, sigma_exp6)
                Ene += vdw
                ELJ += vdw
                if elec:
                    CC = Coulomb2(distsquare, dielec, qa, qb)
                    Ene += CC
                    ECoul += CC
    return Ene, ELJ, ECoul

# Kinetic energy and instantaneous temperature from the velocities
def calc_temp(vel, nat, k, mass):
    v2 = 0.0
    for i in range(len(vel)):
        v2 = v2 + vel[i][0]**2 + vel[i][1]**2
    kin = 0.5*mass*v2          # kinetic energy = 1/2 m v^2
    temp = kin/(nat*k)         # N k T = kinetic energy
    return kin, temp

## 4. Force functions

The force on an atom is minus the gradient of the total potential energy. For the
Lennard-Jones term the component along $x$ is obtained by the chain rule,

$$F_x = -\frac{\partial E}{\partial x} = -\frac{\partial E}{\partial u}\,
        \frac{\partial u}{\partial r}\,\frac{\partial r}{\partial x},
\qquad u=\left(\frac{\sigma}{r}\right)^{6},\quad E_{LJ}=4\varepsilon\,(u^{2}-u),$$

which maps directly onto the code:

* `dedu` $= \partial E/\partial u = 4\varepsilon\,(2u-1)$
* `dudr` $= \partial u/\partial r = -6\,\sigma^{6}/r^{7}$
* `drdx` $= x_i/r$, where `xi` $= x_j - x_i$ — using the $j-i$ difference already carries the
  sign that turns $-\partial E/\partial x_i$ into the force on atom $i$.

The Coulomb force follows the same pattern, with `dedr` $= -q_a q_b/(\epsilon_r\, r^{2})$.

In [ ]:
# LJ force component (uses squared distance)
def ForceLJ2(distsquare, epsilon, sigma_exp6, xi):
    # E_LJ = 4 eps [ (sigma/r)^12 - (sigma/r)^6 ] = 4 eps (u^2 - u) with u = (sigma/r)^6
    rij = sqrt(distsquare)
    u    = (1/distsquare)**3 * sigma_exp6
    dedu = 4*epsilon*(2*u - 1)                # dE/du
    dudr = sigma_exp6*(-6.0/rij**7.0)         # du/dr
    drdx = xi/rij
    return dedu*dudr*drdx

# Coulomb force component (uses squared distance)
def ForceCoulomb2(distsquare, dielec, qa, qb, xi):
    rij = sqrt(distsquare)
    dedr = -1.0*(qa*qb/dielec)*(1/distsquare)
    drdx = xi/rij
    return dedr*drdx

# Total force on each atom from Evdw + Ecoulomb (uses squared distance)
def Calc_Force2(coord, epsilon, sigma, dielec, cutoffsquare, boxdim):
    Force = []
    sigma_exp6 = sigma**6
    for i in range(len(coord)):
        tmpforce = [0.0, 0.0]
        for j in range(len(coord)):
            if i == j:
                continue
            # squared atomic distance (nearest image)
            distsquare = 0
            for k in range(2):
                tmp = coord[j][k] - coord[i][k]
                halfbox = boxdim[k]/2
                tmp = tmp - SignR(halfbox, tmp-halfbox) - SignR(halfbox, tmp+halfbox)
                distsquare += tmp**2
            if distsquare < cutoffsquare:
                qa = coord[i][2]
                qb = coord[j][2]
                fflist = []
                for k in range(2):
                    tmp = coord[j][k] - coord[i][k]
                    ff = ForceLJ2(distsquare, epsilon, sigma_exp6, tmp)
                    ff += ForceCoulomb2(distsquare, dielec, qa, qb, tmp)
                    fflist.append(ff)
                for k in range(2):
                    tmpforce[k] = tmpforce[k] + fflist[k]
        Force.append(tmpforce)
    return Force

## 5. The Verlet integrator

This is the core of the MD engine -- the counterpart of the *minimizer* in the EM notebooks.
Three small functions carry out the time integration, in the order a run uses them:

* **`Step1`** -- the bootstrap. At $t=0$ there is no previous position yet, so this one takes a
  single step from the starting positions using the initial velocities and forces. **Yours to
  complete** -- part **(b)** below.
* **`Verlet`** -- the workhorse, used for every step after the first: given the current
  positions, the previous positions, the forces and the time step `h`, it produces the positions
  one time step later. **Yours to complete** -- part **(c)** below.
* **`CalcVel`** -- reconstructs the velocities from the old and the new positions; these are not
  needed to propagate the system, only to report the kinetic energy and the temperature.

---
### **Exercise -- derive the Verlet integrator, then make it work**

> **How this works.** The two functions above are complete except for the lines that compute the
> new positions, which hold a `TODO` placeholder. Derive the integrator on paper first (part
> **a**), then fill the lines in (parts **b** and **c**) and re-run the cell: it checks each
> function against steps worked out by hand and tells you, line by line, where you stand. Until
> both read `OK`, the run in section 8 stays parked at the starting configuration and the
> sections below it have no trajectory to show. A worked solution is folded away under the code
> cell -- try it yourself first.

**(a) Derive the Verlet integration equations.** Follow the following steps:

**i.** Give the Taylor expansion of the atom positions as a function of time, for a small time
step of $dt$:

**ii.** Give the Taylor expansion of the atom positions as a function of time, for a small time
step of $-dt$:

**iii.** "Combine" these two expressions so that you can calculate the positions of the atoms at
$t+dt$ based on the previous and current force expressions.

Work in one dimension, $x(t)$, and treat $y$ the same way afterwards -- the two coordinates are
independent. Newton's law gives you the acceleration, $a(t) = F(t)/m$, and that is as far as you
can take either expansion: stop at the $dt^2$ term, since nothing in the program knows the third
derivative of the position. When you have an answer, two questions worth asking: which terms
**cancelled** when you combined (i) and (ii), and what did that cancellation buy you -- both in
**accuracy** (what is the leading error term?) and in **what you no longer need to know** in
order to take a step?

**(b) Code the first step.** `Step1` has to get the system from $t=0$ to $t=h$ with no previous
position available, so it is your expansion **(a)(i)** exactly as it stands, driven by the
initial velocities. Replace the two `TODO` placeholders in `Step1`.

**(c) Code the Verlet step.** Replace the two `TODO` placeholders in `Verlet` with your result
from **(a)(iii)**.

Everything either function needs is already unpacked for you inside the loop:

| In the code | Meaning | Where |
|---|---|---|
| `r0x`, `r0y` | the position of atom `i` now: $\mathbf{r}(0)$ in `Step1`, $\mathbf{r}(t)$ in `Verlet` | both |
| `v0x`, `v0y` | its initial velocity $\mathbf{v}(0)$ | `Step1` |
| `oldr0x`, `oldr0y` | its previous position $\mathbf{r}(t-h)$ | `Verlet` |
| `force[i][0]`, `force[i][1]` | the $x$ and $y$ components of the force on it, $\mathbf{F}(t)$ | both |
| `mass` | the particle mass $m$ | both |
| `h` | the time step (the $dt$ of your derivation) | both |

Then re-run this cell and the cells of sections 7-11 (or *Kernel -> Restart & Run All*). The
self-check is only the first hurdle: the real test is the **total energy** plotted in section 9.
If your integrator is right it stays essentially flat, and that is something no amount of algebra
can fake.

In [ ]:
# ---------------------------------------------------------------------------
# The time integration. EXERCISE: the four marked lines are yours to write --
# derive them on paper first (part a), then transcribe them here.
# ---------------------------------------------------------------------------
TODO = float("nan")        # placeholder; it disappears as you fill the lines in

# The very first MD step: from t = 0 to t = h, bootstrapped from the initial
# velocities (there is no previous position to Verlet from yet).
def Step1(atom_coord, velocity, force, h, mass):
    newlist = []
    for i in range(len(atom_coord)):
        q = atom_coord[i][2]
        r0x = atom_coord[i][0]              # starting position r(0)
        r0y = atom_coord[i][1]
        v0x = velocity[i][0]                # initial velocity   v(0)
        v0y = velocity[i][1]
        r0xnew = TODO                       # <-- (b) your line here: x at t = h
        r0ynew = TODO                       # <-- (b) your line here: y at t = h
        newlist.append([r0xnew, r0ynew, q])
    return newlist

# Every step after the first: advance each atom from r(t) to r(t+h).
def Verlet(atom_coord, force, h, old_atom_coord, mass):
    newlist = []
    for i in range(len(atom_coord)):
        q = atom_coord[i][2]
        r0x = atom_coord[i][0]              # current position  r(t)
        r0y = atom_coord[i][1]
        oldr0x = old_atom_coord[i][0]       # previous position r(t-h)
        oldr0y = old_atom_coord[i][1]
        r0xnew = TODO                       # <-- (c) your line here: x at t+h
        r0ynew = TODO                       # <-- (c) your line here: y at t+h
        newlist.append([r0xnew, r0ynew, q])
    return newlist

# Velocities from the old and new positions (output only: kinetic energy, temperature)
def CalcVel(old_atom_coord, atom_coord, h):
    newlist = []
    for i in range(len(atom_coord)):
        r0x = atom_coord[i][0]
        r0y = atom_coord[i][1]
        oldr0x = old_atom_coord[i][0]
        oldr0y = old_atom_coord[i][1]
        v0x = (r0x - oldr0x)/(2*h)
        v0y = (r0y - oldr0y)/(2*h)
        newlist.append([v0x, v0y])
    return newlist

# --- self-check: single-particle steps worked out by hand ------------------
def _check(call, cases):
    """Return None if `call` reproduces every hand-computed case, else why not."""
    for desc, args, wantx, wanty in cases:
        try:
            gotx, goty = call(*args)[0][:2]
            if gotx != gotx or goty != goty:      # NaN: the TODOs are still there
                return "not yet -- the two TODO placeholders are still in place"
            close = abs(gotx - wantx) < 1e-9 and abs(goty - wanty) < 1e-9
        except Exception as exc:
            return "raised %s: %s" % (type(exc).__name__, exc)
        if not close:
            return ("wrong for %s\n             expected (%g, %g), your lines give (%g, %g)"
                    % (desc, wantx, wanty, gotx, goty))
    return None

_step1_cases = [
    ("r(0) = (10, 20), v(0) = (2, -4), F(0) = (4, -8), m = 2, h = 0.5",
     ([[10.0, 20.0, 1.0]], [[2.0, -4.0]], [[4.0, -8.0]], 0.5, 2.0), 11.25, 17.5),
    ("r(0) = (0, 7), v(0) = (3, 2), no force, m = 1, h = 1",
     ([[0.0, 7.0, -1.0]], [[3.0, 2.0]], [[0.0, 0.0]], 1.0, 1.0), 3.0, 9.0),
]
_verlet_cases = [
    ("r(t) = (10, 20), r(t-h) = (8, 23), F(t) = (4, -8), m = 2, h = 0.5",
     ([[10.0, 20.0, 1.0]], [[4.0, -8.0]], 0.5, [[8.0, 23.0, 1.0]], 2.0), 12.5, 16.0),
    ("r(t) = (0, 7), r(t-h) = (-3, 5), no force, m = 1, h = 1",
     ([[0.0, 7.0, -1.0]], [[0.0, 0.0]], 1.0, [[-3.0, 5.0, -1.0]], 1.0), 3.0, 9.0),
]

_why1 = _check(Step1,  _step1_cases)
_why2 = _check(Verlet, _verlet_cases)
INTEGRATOR_READY = _why1 is None and _why2 is None

print("(b) Step1 :", "OK" if _why1 is None else _why1)
print("(c) Verlet:", "OK" if _why2 is None else _why2)
if INTEGRATOR_READY:
    print("\nBoth reproduce the hand-computed positions -- section 8 can integrate.")
else:
    print("\nSection 8 stays at the starting configuration until both lines above read OK.")

<details>
<summary><b>Exercise -- worked solution</b></summary>

**(i) A step forward, $+dt$.** Taylor-expand the position about time $t$:

$$x(t+dt) = x(t) + dt\,v(t) + \frac{dt^2}{2}\,a(t) + \frac{dt^3}{6}\,b(t) + \mathcal{O}(dt^4)$$

with $v=\dot x$ the velocity, $a=\ddot x$ the acceleration and $b=\dddot x$ the third derivative.

**(ii) A step backward, $-dt$.** The same expansion with $dt \to -dt$, so every **odd** power of
$dt$ changes sign:

$$x(t-dt) = x(t) - dt\,v(t) + \frac{dt^2}{2}\,a(t) - \frac{dt^3}{6}\,b(t) + \mathcal{O}(dt^4)$$

**(iii) Combine them -- by adding.** Add (i) and (ii): every odd-power term cancels, taking the
velocity and the third derivative with it:

$$x(t+dt) + x(t-dt) = 2\,x(t) + dt^2\,a(t) + \mathcal{O}(dt^4)$$

Rearrange for the quantity you want, and substitute $a(t) = F(t)/m$ from Newton's law:

$$x(t+dt) = 2\,x(t) - x(t-dt) + \frac{F(t)}{m}\,dt^2 + \mathcal{O}(dt^4)$$

and identically for $y$. That is the Verlet integrator: the next position from the **current**
position, the **previous** position and the **current** force.

Both of the questions at the end of (a) are answered by that cancellation, and together they are
why this scheme -- published by Loup Verlet in 1967 -- is still the backbone of most MD codes:

* **You never need the velocities to advance the system.** One step needs two positions and one
  force, nothing else. Velocities are still useful, but only as *output*: `CalcVel` reconstructs
  them afterwards for the kinetic energy and the temperature.
* **The error per step is $\mathcal{O}(dt^4)$**, not the $\mathcal{O}(dt^3)$ of either expansion
  on its own -- the $dt^3$ terms cancelled along with the velocity, so you gained an order of
  accuracy for free. And because $dt$ now appears only *squared*, the formula is unchanged by
  $dt \to -dt$: the integrator is exactly **time-reversible**, which is why the total energy
  fluctuates but shows no systematic drift over a long run.

**(b) `Step1`.** The first step cannot use (iii) -- there is no $x(t-dt)$ yet -- so it is
expansion (i) itself, cut off after the $dt^2$ term, with $a(0) = F(0)/m$:

```python
r0xnew = r0x + h*v0x + 0.5*h**2*force[i][0]/mass
r0ynew = r0y + h*v0y + 0.5*h**2*force[i][1]/mass
```

This single step is only $\mathcal{O}(h^3)$ accurate -- *worse* than every Verlet step that
follows it -- and it is the one place where the velocities are indispensable. Both are acceptable
because it happens exactly once: from then on there is always a previous position to work with.

**(c) `Verlet`.** A direct transcription of the boxed result above:

```python
r0xnew = 2*r0x - oldr0x + force[i][0]/mass * h**2
r0ynew = 2*r0y - oldr0y + force[i][1]/mass * h**2
```

With both in place the self-check reports `OK` twice and section 8 integrates the full 2000
steps: the total energy in section 9 stays flat to within about 1 % (it drifts from 2786 to 2809
over the run) while kinetic and potential energy visibly trade back and forth, and the mean
temperature comes out at 293 K, close to the 300 K the initial velocities were drawn at.

</details>

## 6. Parameters

Change any of these parameters and re-run **this cell together with the Initialisation and Run cells just
below** to explore their effect (or use *Kernel → Restart & Run All*). Values are in the toy
model's arbitrary units.

> **Note.** The **system** parameters below are the *same* as in the energy-minimization
> notebooks (`nAtoms=20`, `Radius=25`, `Epsilon=6.25`, `qat=Radius`, `Seed=100`, …), so the MD
> starts from the identical configuration those notebooks minimize. MD adds the extra
> **dynamical** parameters `Mass`, `Temperature` and `timestep`.

### System and its properties

| Parameter | Meaning | Typical value / range |
|---|---|---|
| `nAtoms` | number of particles | 2–40 |
| `Radius` | particle radius (drawn size, and default absolute charge) | 10–40 — must leave room to place all atoms |
| `Mass` | particle mass (enters $\mathbf{a}=\mathbf{F}/m$) | 10 |
| `Sigma` | LJ distance parameter $\sigma$: the separation where $E_{LJ}=0$ (the minimum sits at $2^{1/6}\sigma$) | `2.24 * Radius` |
| `BoxDim` | box dimensions (periodic) | `[500, 500]` |
| `Epsilon` | LJ well depth (the minimum of $E_{LJ}$ is $-$`Epsilon`) | 0.25–25 |
| `Dielec` | dielectric constant (charge screening) | 1 (vacuum) – 80 (water) |
| `qat` | absolute charge per atom | defaults to `Radius` |
| `frac_neg` | fraction of negative charges | 0–1 |
| `CutOff` | non-bonded cutoff distance | 250 |

### Molecular dynamics controls

| Parameter | Meaning | Typical value / range |
|---|---|---|
| `Temperature` | target temperature for the initial velocities (K) | 300 |
| `timestep` | integration time step `h` (smaller → better energy conservation) | 5e-3 |
| `cstboltz` | Boltzmann constant (unit conversion) | fixed |
| `Seed` | random-number seed (reproducibility) | 100 |
| `nsteps` | number of MD steps to integrate | 2000 |

In [ ]:
nAtoms  = 20              # number of particles
Radius  = 25.0            # particle radius (must leave room to place all atoms)
Mass    = 10.0            # particle mass
Sigma   = 2.24 * Radius   # LJ distance parameter sigma (E_LJ = 0 here; minimum at 2^(1/6)*Sigma)
BoxDim  = [500, 500]      # box dimensions
Epsilon = 6.25            # LJ well depth (the minimum of E_LJ is -Epsilon)
Dielec  = 1.0             # dielectric constant
qat     = Radius          # atom absolute charge
frac_neg = 0.5            # fraction of negative charges
OverlapFr = 0.0           # fraction of overlap allowed when placing atoms
CutOff  = 250             # non-bonded cutoff
CutOffSquare = CutOff**2

# --- molecular dynamics controls ---
cstboltz = 0.00198722          # Boltzmann constant in kcal/mol/K
cstboltz = 1000*cstboltz/4.18  # -> kJ/mol/K
Temperature = 300.0            # target temperature (K) for the initial velocities
timestep = 5.0e-3              # integration time step h
Seed     = 100                 # random number seed (reproducibility, shared with the EM notebooks)
nsteps   = 2000                # number of MD steps to run

## 7. Initialisation

Generate random, non-overlapping starting positions (a fraction `frac_neg` negative, the rest
positive), then draw **initial velocities** from a Maxwell-Boltzmann distribution at
`Temperature`. The overall (centre-of-mass) motion is removed and the velocities are rescaled
so the initial temperature is exactly the target value.

In [ ]:
import sys

### generate random, non-overlapping coordinates ###
def InitConf(n, dim, radius, qat, frac_neg):
    seed(Seed)
    print("Initializing box, please wait...")
    tmp_coord = []
    i = 0
    ntrial = 0
    nneg = int(float(n) * frac_neg)
    npos = n - nneg

    # first atom
    x = random()*(dim[0]-2*radius) + radius
    y = random()*(dim[1]-2*radius) + radius
    charge = -qat
    if npos == n:
        charge = qat
    i += 1
    if n == 2:
        tmp_coord.append([175, 300, charge])
    else:
        tmp_coord.append([x, y, charge])

    # remaining negative charges
    while i < nneg:
        x = random()*(dim[0]-2*radius) + radius
        y = random()*(dim[1]-2*radius) + radius
        OVERLAP = 1
        for j in range(i):
            if dist(tmp_coord[j], [x, y]) < (1-OverlapFr)*2*radius:
                OVERLAP = 0
        if OVERLAP:
            charge = -qat
            if n == 2:
                tmp_coord.append([325, 300, charge])
            else:
                tmp_coord.append([x, y, charge])
            i += 1
        ntrial += 1
        if ntrial > 100000:
            print("initialisation failed -> reduce radius or number of atoms")
            sys.exit()

    # remaining positive charges
    while i < n:
        x = random()*(dim[0]-2*radius) + radius
        y = random()*(dim[1]-2*radius) + radius
        OVERLAP = 1
        for j in range(i):
            if dist(tmp_coord[j], [x, y]) < (1-OverlapFr)*2*radius:
                OVERLAP = 0
        if OVERLAP:
            charge = qat
            if n == 2:
                tmp_coord.append([325, 300, charge])
            else:
                tmp_coord.append([x, y, charge])
            i += 1
        ntrial += 1
        if ntrial > 10**10:
            print("initialisation failed -> reduce radius or number of atoms")
            sys.exit()
    return tmp_coord


### draw initial velocities from a Maxwell-Boltzmann distribution ###
def InitVel(n, temperature, cstboltz, mass):
    stdev = sqrt(cstboltz*temperature/mass)
    print("Initializing velocities, please wait...")
    tmp_vel = []
    if n == 2:                       # for testing: two atoms start at rest
        tmp_vel = [[0.0, 0.0], [0.0, 0.0]]
    else:
        for i in range(n):
            r1 = random()
            r2 = random()
            # Gaussian-distributed components (Box-Muller-like)
            x1 = sqrt(-2.0*log(r1))*cos(r2)
            x2 = sqrt(-2.0*log(r1))*sin(0.5*r2)
            tmp_vel.append([x1*stdev, x2*stdev])

    # remove overall (centre-of-mass) motion
    vxt = sum(v[0] for v in tmp_vel)
    vyt = sum(v[1] for v in tmp_vel)
    for i in range(n):
        tmp_vel[i][0] -= vxt/float(n)
        tmp_vel[i][1] -= vyt/float(n)

    # rescale so the temperature is exactly the target value
    kin, tt = calc_temp(tmp_vel, n, cstboltz, mass)
    scaling = sqrt(temperature/tt)
    vel = [[v[0]*scaling, v[1]*scaling] for v in tmp_vel]
    return vel


Atom_Coord = InitConf(nAtoms, BoxDim, Radius, qat, frac_neg)
Velocity   = InitVel(nAtoms, Temperature, cstboltz, Mass)
Color = [charge_color(a[2], qat) for a in Atom_Coord]
print(f"Placed {len(Atom_Coord)} atoms and initialised their velocities.")

## 8. Run the molecular dynamics

This performs the molecular dynamics. It needs the `Step1` and `Verlet` functions you
completed in section 5: until the self-check there reports `OK` for both, this cell stays at the starting
configuration and the sections below show a single motionless frame. The first step is bootstrapped with `Step1`; every step
after that applies the `Verlet` update, recovers the velocities with `CalcVel`, and records the
potential, kinetic and total energies, the temperature and a snapshot of the positions.

Periodic boundary conditions are applied by wrapping each just-moved position **together with
its immediate predecessor**, so the position *difference* that Verlet relies on is preserved
across the box edge (velocities are computed *before* wrapping, to avoid spurious jumps). The
run aborts early if the system "explodes" (temperature blowing up), which would signal a time
step that is too large.

In [ ]:
def run_md(Atom_Coord, Velocity):
    """Headless velocity-Verlet MD. Returns trajectory + energy/temperature histories."""
    coord = [list(a) for a in Atom_Coord]
    vel   = [list(v) for v in Velocity]
    h = timestep

    def pbc_pair(a, b):
        """Wrap position `a` into the box, applying the same shift to its predecessor `b`."""
        for pp in range(len(a)):
            for i in range(2):
                if a[pp][i] < 0:
                    a[pp][i] += BoxDim[i]; b[pp][i] += BoxDim[i]
                if a[pp][i] > BoxDim[i]:
                    a[pp][i] -= BoxDim[i]; b[pp][i] -= BoxDim[i]

    # --- first step: Taylor bootstrap from the initial velocities ---
    Force = Calc_Force2(coord, Epsilon, Sigma, Dielec, CutOffSquare, BoxDim)
    old = [list(a) for a in coord]
    new = Step1(coord, vel, Force, h, Mass)     # r(h)

    # record step 0 (initial positions and velocities)
    Ene, EneLJ, EneCoul = Calc_Ene2(coord, Epsilon, Sigma, Dielec, CutOffSquare, BoxDim)
    Kin, T = calc_temp(vel, nAtoms, cstboltz, Mass)
    traj   = [[list(a) for a in coord]]
    Epot   = [Ene];  Elj = [EneLJ];  Ecoul = [EneCoul]
    Ekin   = [Kin];  Etot = [Ene + Kin];  Temp = [T]

    print("step %6d  t=%8.3f  Etot=%8.1f  Ekin=%7.1f  Epot=%7.1f  T=%6.1f" % (0, 0.0, Ene+Kin, Kin, Ene, T))

    pbc_pair(new, old)
    oldp = old
    coord = new

    for step in range(1, nsteps+1):
        Force = Calc_Force2(coord, Epsilon, Sigma, Dielec, CutOffSquare, BoxDim)
        nw  = Verlet(coord, Force, h, oldp, Mass)   # next positions
        vel = CalcVel(oldp, nw, h)                  # velocity at the current step

        Ene, EneLJ, EneCoul = Calc_Ene2(coord, Epsilon, Sigma, Dielec, CutOffSquare, BoxDim)
        Kin, T = calc_temp(vel, nAtoms, cstboltz, Mass)

        traj.append([list(a) for a in coord])
        Epot.append(Ene); Elj.append(EneLJ); Ecoul.append(EneCoul)
        Ekin.append(Kin); Etot.append(Ene + Kin); Temp.append(T)

        if T > 1.0e6:
            print("The system is exploding -- emergency stop at step", step)
            break

        pbc_pair(nw, coord)     # wrap next positions together with the current ones
        oldp = coord
        coord = nw

        if step % 100 == 0:
            print("step %6d  t=%8.3f  Etot=%8.1f  Ekin=%7.1f  Epot=%7.1f  T=%6.1f"
                  % (step, step*h, Ene+Kin, Kin, Ene, T))

    time = [i*h for i in range(len(traj))]
    return traj, time, Epot, Elj, Ecoul, Ekin, Etot, Temp


if INTEGRATOR_READY:
    traj, time, Epot, Elj, Ecoul, Ekin, Etot, Temp = run_md(Atom_Coord, Velocity)
    print(f"\nDone: {len(traj)-1} MD steps.  Mean temperature ~ {sum(Temp)/len(Temp):.1f} K")
else:
    # The integrator of section 5 is still incomplete, so there is nothing to integrate:
    # record step 0 only, which keeps the sections below runnable (one motionless frame).
    Ene, EneLJ, EneCoul = Calc_Ene2(Atom_Coord, Epsilon, Sigma, Dielec, CutOffSquare, BoxDim)
    Kin, T = calc_temp(Velocity, nAtoms, cstboltz, Mass)
    traj = [[list(a) for a in Atom_Coord]];  time = [0.0]
    Epot = [Ene];  Elj = [EneLJ];  Ecoul = [EneCoul]
    Ekin = [Kin];  Etot = [Ene + Kin];  Temp = [T]
    print("No trajectory yet -- complete Step1 and Verlet in section 5, then re-run this cell.")

## 9. Energy and temperature

The left panel shows how the **potential**, **kinetic** and **total** energies evolve. Kinetic
and potential energy exchange continuously as atoms speed up and slow down, while the **total**
energy stays essentially flat — the signature of a well-behaved (energy-conserving) integrator.
The right panel shows the instantaneous temperature fluctuating around its mean.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(time, Etot, label="E$_{tot}$", lw=2)
ax1.plot(time, Ekin, label="E$_{kin}$", lw=1.3)
ax1.plot(time, Epot, label="E$_{pot}$", lw=1.3)
ax1.set_xlabel("time")
ax1.set_ylabel("energy")
ax1.set_title("Energy vs time (Verlet MD)")
ax1.legend()
ax1.grid(alpha=0.3)

Tmean = sum(Temp)/len(Temp)
ax2.plot(time, Temp, lw=1.0, color="tab:red")
ax2.axhline(Tmean, ls="--", color="black", lw=1, label=f"mean = {Tmean:.0f} K")
ax2.set_xlabel("time")
ax2.set_ylabel("temperature (K)")
ax2.set_title("Instantaneous temperature")
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Visualise the system

Start and end configurations side by side. White = positive charge, dark = negative.

In [ ]:
def draw_config(ax, coord, title):
    ax.set_xlim(0, BoxDim[0])
    ax.set_ylim(0, BoxDim[1])
    ax.set_aspect('equal')
    ax.set_facecolor("#ccddff")
    ax.set_title(title)
    ax.invert_yaxis()   # match the original canvas (y downwards)
    for a in coord:
        col = charge_color(a[2], qat)
        ax.add_patch(Circle((a[0], a[1]), Radius, facecolor=col, edgecolor="black", lw=0.8))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 5.5))
draw_config(ax1, traj[0],  f"Initial  (t = {time[0]:.2f})")
draw_config(ax2, traj[-1], f"Final  (t = {time[-1]:.2f})")
plt.tight_layout()
plt.show()

## 11. Animation of the trajectory

Replays the whole trajectory.
(To keep it light, only every few frames are shown — adjust `stride`.)

In [ ]:
stride = max(1, len(traj)//120)   # cap at ~120 frames
frames = list(range(0, len(traj), stride))

fig, ax = plt.subplots(figsize=(6, 6))
ax.set_xlim(0, BoxDim[0])
ax.set_ylim(0, BoxDim[1])
ax.set_aspect('equal')
ax.set_facecolor("#ccddff")
ax.invert_yaxis()

circles = [Circle((a[0], a[1]), Radius,
                  facecolor=charge_color(a[2], qat), edgecolor="black", lw=0.8)
           for a in traj[0]]
for c in circles:
    ax.add_patch(c)
title = ax.set_title("")

def update(frame_idx):
    f = frames[frame_idx]
    for c, a in zip(circles, traj[f]):
        c.center = (a[0], a[1])
    title.set_text(f"step {f}   t = {time[f]:.2f}   E$_{{tot}}$ = {Etot[f]:.0f}   T = {Temp[f]:.0f} K")
    return circles + [title]

anim = FuncAnimation(fig, update, frames=len(frames), interval=80, blit=False)
plt.close(fig)   # avoid a duplicate static figure
anim

## 12. Molecular dynamics vs energy minimization

This notebook **integrates the equations of motion**; the companion notebooks
(`LJ-ELEC_EM-steepest`, `LJ-ELEC_EM-conjugate`, `LJ-ELEC_EM-simplex`) instead **minimize** the
energy of the same kind of system. The key differences:

* **Energy minimization** only ever moves *downhill* and halts at the first nearby local
  minimum — it answers *"what is a stable arrangement close to where I started?"*.
* **Molecular dynamics** gives the particles kinetic energy, so they can climb **over energy
  barriers** and keep exploring. It conserves the **total** energy (potential + kinetic) rather
  than driving it to a minimum, and it produces a **time-dependent trajectory** from which one
  can measure temperature, diffusion, and thermodynamic averages.
* Because MD samples many configurations at finite temperature, it does **not** get permanently
  trapped in the first minimum — cooling an MD run down (lowering `Temperature`) is in fact one
  simple way to search for deeper minima than a single minimization would find.

Try lowering `Temperature`, shrinking `timestep` (better energy conservation), or increasing
`nsteps`, and re-run the Parameters → Initialisation → Run cells to see the effect.